Checking GPU Requirements

In [3]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")


PyTorch Version: 2.14.1+cu130
GPU Detected: NVIDIA GeForce RTX 5060 Laptop GPU


Verify BitsAndBytes 4-Bit Quantization Compatibility

In [4]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("BitsAndBytes 4-bit configuration successfully initialized!")

/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BitsAndBytes 4-bit configuration successfully initialized!


Load And Split Dataset

In [ ]:
import json
import random
from datasets import Dataset, DatasetDict

dataset_path = "tanglish.jsonl"

# 1. Load raw dataset
with open(dataset_path, "r", encoding="utf-8") as f:
    raw_records = [json.loads(line) for line in f]

print(f"Loaded {len(raw_records)} records from {dataset_path}")

# 2. Format records into Gemma instruction template
def format_record(record):
    instruction = record.get("instruction", "").strip()
    inp = record.get("input", "").strip()
    output = record.get("output", "").strip()
    
    if inp:
        user_msg = f"{instruction}\n\nContext:\n{inp}"
    else:
        user_msg = instruction
        
    formatted_text = (
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )
    return {"text": formatted_text}

formatted_data = [format_record(r) for r in raw_records]

# 3. Create 90/10 train-validation split
random.seed(42)
random.shuffle(formatted_data)

split_idx = int(len(formatted_data) * 0.9)
train_list = formatted_data[:split_idx]
val_list = formatted_data[split_idx:]

dataset = DatasetDict({
    "train": Dataset.from_list(train_list),
    "validation": Dataset.from_list(val_list)
})

print(f"Split Complete: {len(dataset['train'])} Train | {len(dataset['validation'])} Validation")
print("\n--- SAMPLE FORMATTED PROMPT ---")
print(dataset['train'][0]['text'])

Loaded 613 records from tanglish.jsonl
✅ Split Complete: 551 Train | 62 Validation

--- SAMPLE FORMATTED PROMPT ---
<start_of_turn>user
What is a Higher-Order Component (HOC) in React?

Context:
function withLogger(Component) {
  return function Wrapped(props) {
    console.log('Rendering', Component.name);
    return <Component {...props} />;
  };
}<end_of_turn>
<start_of_turn>model
`HOC` oru function, adhu oru component-a eduthukittu, extra functionality-oda oru `new component`-a return pannum. Example: `withAuth`, `withLogger`, Redux-oda `connect()`. Cross-cutting logic-a (auth check, logging) reuse panna idhu use aagum. Aana ippo `custom hooks` vandhadhukku appuram HOCs-oda usage kammi aagirukku, because hooks simple-a and nested wrappers ('wrapper hell') illama irukkum.<end_of_turn>
